# **Notebook**: Census Tracts Master Database

__Content__: Code utilized to create the Census Tracts Master Database

#### Initial setup

In [1]:
'''Load and import necessary libraries'''
# General data libraries
import numpy as np
import pandas as pd
import geopandas as gpd

import os

# Define data path
data_path = "..."

#### Load data

In [ ]:
''' Load datasets'''
# Load Landslide Exposure Database
led_inventory = gpd.read_file(os.path.join(data_path, "process_building_inventory", "US_LandslideExposureDatabase_points.gpkg"))
print(f'Landslide Exposure Database records: {len(led_inventory)}')

# load census tracts
census_tracts = gpd.read_file(os.path.join(data_path, "census_gov_data", "cb_2024_us_tract_500k.zip"))
print(f'Census tracts: {len(census_tracts)}')

# load CRE data
cre_data = pd.read_csv(os.path.join(data_path, "census_gov_data", "CRE_23_Tract.csv"), encoding="ISO-8859-1")
print(f'CRE data records: {len(cre_data)}')

# load ACS data
acs_data_poverty = pd.read_csv(os.path.join(data_path, "census_gov_data", "ACS_2022_5YR_TRACT/extracted_tables/ACS_2022_X17_POVERTY_combined.csv"), encoding="ISO-8859-1")
acs_data_income = pd.read_csv(os.path.join(data_path, "census_gov_data", "ACS_2022_5YR_TRACT/extracted_tables/ACS_2022_X19_INCOME_combined.csv"), encoding="ISO-8859-1")
print(f'ACS poverty data records: {len(acs_data_poverty)}')
print(f'ACS income data records: {len(acs_data_income)}')

# load FFIEC data
ffiec_data = pd.read_csv(os.path.join(data_path, "census_gov_data", "FFIEC_Census_Tract_List_2023.csv"), encoding="ISO-8859-1")
print(f'FFIEC data records: {len(ffiec_data)}')

# Load Results from LISA cluster analysis
lisa_results = gpd.read_file(os.path.join(data_path, "process_building_inventory", "LISA_cluster_LED-CRE_truncated.gpkg"))
print(f'LISA cluster results records: {len(lisa_results)}')

## 1) Socioeconomic Data

In [ ]:
''' 
Merge data to census tracts using GEOIDFQ
GEOIDFQ is the 15-digit unique identifier for census tracts, which includes state, county, and tract codes.
'''

# Merge CRE data
cre_data = cre_data.rename(columns={'GEO_ID': 'GEOIDFQ'})
census_tracts = census_tracts.merge(cre_data[['GEOIDFQ', 
                                              'PRED0_E', 'PRED0_M', 'PRED0_PE', 'PRED0_PM', 
                                              'PRED12_E','PRED12_M','PRED12_PE', 'PRED12_PM',
                                              'PRED3_E','PRED3_M','PRED3_PE', 'PRED3_PM']], on='GEOIDFQ', how='left')
census_tracts = census_tracts.drop(columns=['index_right'], errors='ignore')

# Merge ACS poverty data
census_tracts = census_tracts.merge(acs_data_poverty[['GEOIDFQ','B17017_E002','B17017_E001']], on='GEOIDFQ', how='left')
census_tracts['POVERTY_RATE'] = census_tracts['B17017_E002'] / census_tracts['B17017_E001']
census_tracts = census_tracts.drop(columns=['index_right'], errors='ignore')

# Merge ACS income data
census_tracts = census_tracts.merge(acs_data_income[['GEOIDFQ','B19013_E001']], on='GEOIDFQ', how='left')
census_tracts = census_tracts.drop(columns=['index_right'], errors='ignore')

# Merge FFIEC data
census_tracts['FIPS_code'] = census_tracts['GEOIDFQ'].str[:11]
# FIPS code in ffiec_data is integer, so we need to convert it to string and pad with zeros to 11 characters
ffiec_data['FIPS code'] = ffiec_data['FIPS code'].astype(str).str.zfill(11)
# rename FIPS code in ffiec data to match building_inventory_pop
ffiec_data = ffiec_data.rename(columns={'FIPS code': 'FIPS_code'})
census_tracts = census_tracts.merge(ffiec_data[['FIPS_code', 'Tract MFI', 'Tract income percentage', 'Tract income level']], on='FIPS_code', how='left')
census_tracts = census_tracts.drop(columns=['index_right', 'FIPS_code'], errors='ignore')
census_tracts = census_tracts.rename(columns={'Tract MFI': 'TRACT_MFI', 'Tract income percentage': 'TRACT_INCOME_PCT', 'Tract income level': 'Tract_income_level'})

## 2) Building Information Aggregation

In [ ]:
# Aggregate landslide exposure data to census tract level
led_tract_agg = led_inventory.groupby('GEOIDFQ').agg(
    total_buildings = ('geoid', 'count'),
    high_exp_bld = led_inventory['susc_class'].apply(lambda x: (x == 'high').count()),
    moderate_exp_bld = led_inventory['susc_class'].apply(lambda x: (x == 'moderate').count()),
    low_exp_bld = led_inventory['susc_class'].apply(lambda x: (x == 'low').count()),
    none_exp_bld = led_inventory['susc_class'].apply(lambda x: (x == 'none').count()),
    high_exp_pop = led_inventory[led_inventory['susc_class'] == 'high']['pop_night'].sum(),
    moderate_exp_pop = led_inventory[led_inventory['susc_class'] == 'moderate']['pop_night'].sum(),
    low_exp_pop = led_inventory[led_inventory['susc_class'] == 'low']['pop_night'].sum(),
    none_exp_pop = led_inventory[led_inventory['susc_class'] == 'none']['pop_night'].sum()
).reset_index()
census_tracts = census_tracts.merge(led_tract_agg, on='GEOIDFQ', how='left')
census_tracts = census_tracts.drop(columns=['index_right'], errors='ignore')

#### Save data

In [ ]:
census_tracts.to_file(os.path.join(data_path, "process_building_inventory", "US_CensusTractsMaster.gpkg"), driver="GPKG")

#### End of Script